# 04 - Final Emotion Model and Combined NLP Analysis

This backend-free notebook trains the six-class emotion model, evaluates it on fixed held-out splits, and combines it with the sentiment model and transparent topic rules. It reuses the tested code in `ai/nlp/`; it does not duplicate model logic.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from sklearn.model_selection import train_test_split

AI_DIR = Path.cwd().resolve().parent
sys.path.insert(0, str(AI_DIR))

from nlp.training import EMOTION_LABELS, RANDOM_STATE, save_model, train_classifier

EMOTION_DATA = AI_DIR / 'datasets' / 'emotions.csv'
EMOTION_MODEL_DIR = AI_DIR / 'models' / 'emotion'
SENTIMENT_MODEL = AI_DIR / 'models' / 'sentiment' / 'sentiment_tfidf_logreg.joblib'
MAX_ROWS = 60_000  # Set to 0 only when enough RAM is available for all rows.


In [ ]:
raw = pd.read_csv(EMOTION_DATA)
if set(raw.columns) != {'text', 'label'}:
    raise ValueError("Expected exactly 'text' and 'label' columns.")

emotion_df = raw.rename(columns={'text': 'feedback_text'}).copy()
emotion_df['emotion'] = emotion_df['label'].map(dict(enumerate(EMOTION_LABELS)))
if emotion_df['emotion'].isna().any():
    raise ValueError('Found an emotion label outside 0-5.')

if MAX_ROWS and len(emotion_df) > MAX_ROWS:
    emotion_df, _ = train_test_split(
        emotion_df, train_size=MAX_ROWS, random_state=RANDOM_STATE, stratify=emotion_df['emotion']
    )

emotion_df['emotion'].value_counts().rename_axis('emotion').to_frame('rows')


In [ ]:
train_df, holdout_df = train_test_split(
    emotion_df, test_size=0.20, random_state=RANDOM_STATE, stratify=emotion_df['emotion']
)
validation_df, test_df = train_test_split(
    holdout_df, test_size=0.50, random_state=RANDOM_STATE, stratify=holdout_df['emotion']
)

emotion_model, emotion_result = train_classifier(
    train_df, validation_df, test_df, label_column='emotion', tune=False
)
model_path, metrics_path = save_model(
    emotion_model, emotion_result, EMOTION_MODEL_DIR, 'emotion_tfidf_logreg'
)
print(f'Validation macro F1: {emotion_result.validation_macro_f1:.3f}')
print(f'Test macro F1: {emotion_result.test_macro_f1:.3f}')
print(f'Saved model: {model_path}')
print(f'Saved metrics: {metrics_path}')


## Final NLP check

Run notebook 03 or `python training/train_sentiment.py` first so the local sentiment artifact exists. Topic labels are reviewable keyword matches, while sentiment and emotion include calibrated class probabilities from their logistic-regression pipelines.

In [ ]:
from nlp import FeedbackAnalyzer

if not SENTIMENT_MODEL.exists():
    raise FileNotFoundError('Train the sentiment model in notebook 03 before this step.')

analyzer = FeedbackAnalyzer(SENTIMENT_MODEL, model_path)
analyzer.analyze(
    'The professor explains difficult topics clearly, but the assignment deadline is stressful.'
)
